In [24]:
import pandas as pd
df=pd.read_csv('dalariesDS.csv')
enflation={
    2020:1.25,
    2021:1.2,
    2022:1.11,
    2023:1.07
}
df['salary_in_usd_2026']=df['salary_in_usd']* df.work_year.map(enflation)
x=df[['experience_level','employment_type','job_title','employee_residence','remote_ratio','company_location','company_size']]
y=df.iloc[:,-1]
def train_test_validation(x,y,test_size=0.25,random_state=12):
    from sklearn.model_selection import train_test_split;x_,x_test,y_,y_test=train_test_split(x,y,test_size=test_size,random_state=random_state);x_train,x_val,y_train,y_val=train_test_split(x_,y_,test_size=(test_size/(1-test_size)),random_state=random_state);return x_train,x_test,x_val,y_train,y_test,y_val
x_train,x_test,x_val,y_train,y_test,y_val=train_test_validation(x,y)

In [26]:
nominal_cols=['job_title','employee_residence','company_location']

ordinal_cols=['experience_level','employment_type','remote_ratio','company_size']
ordinal_categories=[
    ['EN','MI','SE','EX'],
    ['PT','FL','CT','FT'],
    [0,50,100],
    ['S','M','L']    
]

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder,OrdinalEncoder
from sklearn.compose import ColumnTransformer
ordinal_pipeline=Pipeline(steps=[
    ('Ordinal encoding',OrdinalEncoder(categories=ordinal_categories,handle_unknown='use_encoded_value',unknown_value=-1))
])
nominal_pipeline=Pipeline(steps=[
    ('OneHotEncoder',OneHotEncoder(handle_unknown='ignore'))
])
ct=ColumnTransformer(transformers=[
    ('ordinal encoding',ordinal_pipeline,ordinal_cols),
    ('hot encoding',nominal_pipeline,nominal_cols)
])

In [42]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import make_pipeline
pipe1=make_pipeline(ct,DecisionTreeRegressor())
pipe1.fit(x_train,y_train)
result_dtr=pipe1.score(x_val,y_val)   # 0.3035469026004235

pipe2=make_pipeline(ct,RandomForestRegressor())
pipe2.fit(x_train,y_train)
result_rfr=pipe2.score(x_val,y_val)  #0.35899648214014934

In [ ]:
from sklearn.model_selection import GridSearchCV
params=[
    {'decisiontreeregressor__max_depth': [None, 3, 5, 10, 15, 20, 30],
    'decisiontreeregressor__min_samples_split': [2, 5, 10, 20],
    'decisiontreeregressor__min_samples_leaf': [1, 2, 4, 10],
    'decisiontreeregressor__max_features': [1.0, 'sqrt', 'log2']},
    {
    'randomforestregressor__n_estimators': [100, 200, 300, 500],
    'randomforestregressor__max_depth': [None, 5, 10, 20, 30],
    'randomforestregressor__min_samples_split': [2, 5, 10],
    'randomforestregressor__min_samples_leaf': [1, 2, 4],
    'randomforestregressor__max_features': [1.0, 'sqrt', 'log2']
}
]

grid_search1=GridSearchCV(estimator=pipe1,param_grid=params[0],verbose=0,cv=4,n_jobs=-1,scoring='r2')
grid_search1.fit(x_train,y_train)
print(f"FOR DTR:\nBest parameters: {grid_search1.best_params_}\nBest score: {grid_search1.best_score_}")
#!-------------------------------------------------------------------------------------------------------
grid_search2=GridSearchCV(estimator=pipe2,param_grid=params[1],verbose=0,cv=4,n_jobs=-1,scoring='r2')
grid_search2.fit(x_train,y_train)
print(f"FOR RFR:\nBest parameters: {grid_search2.best_params_}\nBest score: {grid_search2.best_score_}")


FOR DTR:
Best parameters: {'decisiontreeregressor__max_depth': 20, 'decisiontreeregressor__max_features': 1.0, 'decisiontreeregressor__min_samples_leaf': 4, 'decisiontreeregressor__min_samples_split': 20}
Best score: 0.37763228668764354
FOR RFR:
Best parameters: {'randomforestregressor__max_depth': 30, 'randomforestregressor__max_features': 'sqrt', 'randomforestregressor__min_samples_leaf': 1, 'randomforestregressor__min_samples_split': 10, 'randomforestregressor__n_estimators': 300}
Best score: 0.4162024146823441


In [54]:
from sklearn.metrics import r2_score,mean_absolute_error
dtr=grid_search1.best_estimator_
rfr=grid_search2.best_estimator_

print(f'DTR r2 score test: {r2_score(y_test,dtr.predict(x_test))}\nRFR r2 score test: {r2_score(y_test,rfr.predict(x_test))}')
print(f"DTR MeanAbsError: ${mean_absolute_error(y_test,dtr.predict(x_test)):,.0f}\nRFR MeanAbsError: ${mean_absolute_error(y_test,rfr.predict(x_test)):,.0f}")

DTR r2 score test: 0.39161340133899025
RFR r2 score test: 0.41798766787382935
DTR MeanAbsError: $40,585
RFR MeanAbsError: $39,484
